# **1. Importação das bibliotecas**

In [ ]:
import numpy as np
import math
import re
import pandas as pd
from bs4 import BeautifulSoup
import zipfile
import seaborn as sns
import spacy as sp
import string
import random
import matplotlib.pyplot as plt

In [ ]:
import tensorflow as tf
tf.__version__

'2.15.0'

In [ ]:
from tensorflow.keras import layers
import tensorflow_datasets as tfds

# **2. Carregamento dos dados**

In [ ]:
cols = ['sentiment', 'id', 'date', 'query', 'user', 'text']

In [ ]:
train_data = pd.read_csv('/content/drive/MyDrive/trainingandtestdata/train.csv',
                          header = None, names = cols, engine = 'python',
                          encoding = 'latin1')

In [ ]:
train_data.shape

(1600000, 6)

In [ ]:
train_data.head()

,sentiment,id,date,query,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [ ]:
train_data.tail()

,sentiment,id,date,query,user,text
1599995,4,2193601966,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,AmandaMarie1028,Just woke up. Having no school is the best fee...
1599996,4,2193601969,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,TheWDBoards,TheWDB.com - Very cool to hear old Walt interv...
1599997,4,2193601991,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,bpbabe,Are you ready for your MoJo Makeover? Ask me f...
1599998,4,2193602064,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,tinydiamondz,Happy 38th Birthday to my boo of alll time!!! ...
1599999,4,2193602129,Tue Jun 16 08:40:50 PDT 2009,NO_QUERY,RyanTrevMorris,happy #charitytuesday @theNSPCC @SparksCharity...


In [ ]:
train_data.sentiment.unique()

array([0, 4])

In [ ]:
test_data = pd.read_csv('/content/drive/MyDrive/trainingandtestdata/test.csv',
                          header = None, names = cols, engine = 'python',
                          encoding = 'latin1')

In [ ]:
test_data.shape

(498, 6)

In [ ]:
test_data.head()

,sentiment,id,date,query,user,text
0,4,3,Mon May 11 03:17:40 UTC 2009,kindle2,tpryan,@stellargirl I loooooooovvvvvveee my Kindle2. ...
1,4,4,Mon May 11 03:18:03 UTC 2009,kindle2,vcu451,Reading my kindle2... Love it... Lee childs i...
2,4,5,Mon May 11 03:18:54 UTC 2009,kindle2,chadfu,"Ok, first assesment of the #kindle2 ...it fuck..."
3,4,6,Mon May 11 03:19:04 UTC 2009,kindle2,SIX15,@kenburbary You'll love your Kindle2. I've had...
4,4,7,Mon May 11 03:21:41 UTC 2009,kindle2,yamarama,@mikefish Fair enough. But i have the Kindle2...


In [ ]:
test_data.tail()

,sentiment,id,date,query,user,text
493,2,14072,Sun Jun 14 04:31:43 UTC 2009,latex,proggit,Ask Programming: LaTeX or InDesign?: submitted...
494,0,14073,Sun Jun 14 04:32:17 UTC 2009,latex,sam33r,"On that note, I hate Word. I hate Pages. I hat..."
495,4,14074,Sun Jun 14 04:36:34 UTC 2009,latex,iamtheonlyjosie,Ahhh... back in a *real* text editing environm...
496,0,14075,Sun Jun 14 21:36:07 UTC 2009,iran,plutopup7,"Trouble in Iran, I see. Hmm. Iran. Iran so far..."
497,0,14076,Sun Jun 14 21:36:17 UTC 2009,iran,captain_pete,Reading the tweets coming out of Iran... The w...


In [ ]:
test_data.sentiment.unique()

array([4, 0, 2])

In [ ]:
data = train_data

In [ ]:
data.drop(['id', 'date', 'query', 'user'], axis=1, inplace=True)

In [ ]:
data.head()

,sentiment,text
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,is upset that he can't update his Facebook by ...
2,0,@Kenichan I dived many times for the ball. Man...
3,0,my whole body feels itchy and like its on fire
4,0,"@nationwideclass no, it's not behaving at all...."


In [ ]:
X = data.iloc[:, 1].values

In [ ]:
X.shape, type(X)

((1600000,), numpy.ndarray)

In [ ]:
y = data.iloc[:, 0].values

In [ ]:
y.shape, type(y)

((1600000,), numpy.ndarray)

# **3. Limpeza e tratamento**

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X, _, y, _ = train_test_split(X, y, test_size=0.85, stratify=y)

In [ ]:
X.shape, y.shape

((240000,), (240000,))

In [ ]:
unique, counts = np.unique(y, return_counts=True)
unique, counts

(array([0, 4]), array([120000, 120000]))

In [ ]:
nlp = sp.load("en_core_web_sm")
stop_words = sp.lang.en.STOP_WORDS

In [ ]:
print(stop_words)

{'a', 'them', 'just', 'we', 'now', 'no', 'onto', 'same', 'are', 'per', 'nevertheless', "'s", 'seems', 'nine', 'whether', 'somehow', 'bottom', 'thereby', '’m', 'whereas', 'around', 'our', 'and', 'five', '‘m', 'their', 'there', 'must', 'seeming', 'thereafter', 'yourselves', "'re", 'quite', 'eleven', 'various', 'than', 'many', 'though', 'last', 'here', 'one', 'full', 'everywhere', 'become', 'has', 'upon', '’s', 'see', 'own', 'some', 'with', 'nothing', 'everything', 'ever', 'otherwise', 'what', 'she', 'then', 'wherever', 'someone', 'afterwards', 'did', 'indeed', 'side', 'were', 'so', 'when', 'fifteen', 'was', 'whose', 'sixty', 'say', "'ve", 'above', 'became', 'most', "'ll", 'keep', 'next', 'none', 'unless', 'two', 'nobody', 'his', '’ll', 'sometime', 'being', 'others', 'who', 'however', 'while', 'below', 'both', 'himself', "'m", 'neither', 'make', 'twelve', 'which', 'thru', 'under', 'whatever', 'namely', 'your', 'until', 'other', 'enough', 'not', 'latter', 'something', 'using', 'would', 'bu

In [ ]:
len(stop_words)

326

In [ ]:
def clean_tweet(tweet):
  tweet = BeautifulSoup(tweet, "lxml").get_text()


  tweet = re.sub(r"@ [A-Za-z0-9]+", " ", tweet)
  tweet = re.sub(r"https?:// [A-Za-z0-9./]+", " ", tweet)
  tweet = re.sub(r"[^a-zA-Z.!?]", " ", tweet)
  tweet = re.sub(r" +", " ", tweet)

  tweet = tweet.lower()
  document = nlp(tweet)

  words = []

  for token in document:
    words.append(token.text)

  words = [word for word in words if word not in stop_words and word not in string.punctuation]
  words = " ".join([str(element) for element in words])

  return words

In [ ]:
data_clean = [clean_tweet(tweet) for tweet in X]

<ipython-input-29-aded9c7cc950>:2: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  tweet = BeautifulSoup(tweet, "lxml").get_text()


In [ ]:
for _ in range(10):
  print(data_clean[random.randint(0, len(data_clean) -1 )])

db birthday hot
  people m bored
ahhh m awake mean psychology exam soon
  samiyusuf sir great inshaa allah
missing boyfriend s florida m texas ll tomorrow inseperable travling
obtained parts need r exhaust repair ... s day project ... weekend suppose nice
  nere thank m surprised spain background tage awsome amazing performance pretty singer
need buy new workout video t find old
weather interfering running schedule damnit join gym
dollars pay tank want damn u adult responsibilities


In [ ]:
data_labels = y
data_labels[data_labels == 4] = 1

In [ ]:
data_labels

array([1, 0, 1, ..., 0, 1, 0])

In [ ]:
np.unique(data_labels)

array([0, 1])

# **4. Tokenização**

In [ ]:
tokenizer = tfds.deprecated.text.SubwordTextEncoder.build_from_corpus(
    data_clean, target_vocab_size=2**16)

In [ ]:
tokenizer.vocab_size

65782

In [ ]:
print(tokenizer.subwords)

In [ ]:
ids = tokenizer.encode('I love you')
ids

[65599, 65558, 14, 3811]

In [ ]:
text = tokenizer.decode(ids)
text

'I love you'

In [ ]:
data_inputs = [tokenizer.encode(sentense) for sentense in data_clean]
data_inputs

In [ ]:
for _ in range(10):
  print(data_inputs[random.randint(0, len(data_inputs) -1 )])

[138, 4917]
[1, 26813, 592, 2485, 4097, 60, 65642]
[1, 17298, 14313, 65558, 22, 24, 183, 283, 28765]
[120, 3714, 508, 534, 726]
[1, 16227, 6186, 42, 479, 51297, 2753, 1008]
[1, 21116, 110, 212]
[18, 66, 31, 21, 220, 368, 156]
[1, 22734, 2087, 505, 178, 65572, 15467, 10657, 27097, 65572, 17, 297, 1760, 120, 633, 2910, 13805, 86, 43288, 1944]
[1, 6953, 5451, 3, 61, 17591, 37503, 6215, 263]
[7, 47, 3025]


# **5. Padding**

In [ ]:
max_len = max([len(sentence) for sentence in data_inputs]) # pegando a sentença com o maior número de palavras
max_len

51

In [ ]:
data_inputs = tf.keras.preprocessing.sequence.pad_sequences(data_inputs,value=0, # preenchendo as menores sentenças com 0 para ficarem todas do mesmo
                                                            padding='post',      # tamanho da maior sentença
                                                            maxlen= max_len)

In [ ]:
for _ in range(10):
  print(data_inputs[random.randint(0, len(data_inputs) -1 )])

In [ ]:
train_inputs, test_inputs, train_labels, test_labels = train_test_split(
    data_inputs, data_labels, test_size=0.3, stratify=data_labels)

In [ ]:
train_inputs.shape, train_labels.shape

((168000, 51), (168000,))

In [ ]:
test_inputs.shape, test_labels.shape

((72000, 51), (72000,))

# **6. Construção do modelo**

In [ ]:
class DCNN(tf.keras.Model):
  def __init__(self, vocab_size:int, emb_dim:int=128, nb_filters:int=50,
               ffn_units:int=512, nb_classes:int=2, dropout_rate:float=0.1,
               training:bool=False, name:str='dcnn'):
    super(DCNN, self).__init__(name=name)

    self.embedding = layers.Embedding(vocab_size, emb_dim)

    # Camadas de convolução do modelo
    self.bigram = layers.Conv1D(
        filters=nb_filters, kernel_size=2, padding='same', activation='relu')

    self.trigram = layers.Conv1D(
        filters=nb_filters, kernel_size=3, padding='same', activation='relu')

    self.fourgram = layers.Conv1D(
        filters=nb_filters, kernel_size=4, padding='same', activation='relu')

    # Camada de pooling
    self.pool = layers.GlobalMaxPool1D()

    # Camada densa
    self.dense_1 = layers.Dense(units=ffn_units, activation='relu')
    self.dropout= layers.Dropout(rate=dropout_rate)

    if nb_classes == 2:
      self.last_dense = layers.Dense(units=1, activation='sigmoid')
    else:
      self.last_dense = layers.Dense(units=nb_classes, activation='softmax')

  def call(self, inputs, training):
    x = self.embedding(inputs)
    x1 = self.bigram(x)
    x1 = self.pool(x1)
    x2 = self.trigram(x)
    x2 = self.pool(x2)
    x3 = self.fourgram(x)
    x3 = self.pool(x3)

    merget = tf.concat([x1, x2, x3], axis=-1) # (batch_size, 3 * nb_classes)
    merget = self.dense_1(merget)
    merget = self.dropout(merget, training)
    output = self.last_dense(merget)

    return output

In [ ]:
vocab_size = tokenizer.vocab_size
emb_dim = 200
nb_filters = 100
ffn_units = 256
batch_size = 32
nb_classes = len(set(train_labels)) # set para retornar apenas valores únicos
dropout_rate=0.2
epochs=5

# **7. Treinamento do modelo**

In [ ]:
Dcnn = DCNN(vocab_size=vocab_size, emb_dim=emb_dim, nb_filters=nb_filters,
            ffn_units=ffn_units, nb_classes=nb_classes, dropout_rate=dropout_rate)

In [ ]:
if nb_classes == 2:
  Dcnn.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
else:
  Dcnn.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

In [ ]:
check_point_path = "./" # caminho para salvar o treinamento do modelo na raiz do google drive
ckpt = tf.train.Checkpoint(Dcnn=Dcnn)
ckpt_manager = tf.train.CheckpointManager(ckpt, check_point_path, max_to_keep=5)

if ckpt_manager.latest_checkpoint:
  ckpt.restore(ckpt_manager.latest_checkpoint)
  print("Latest check point restored")

In [ ]:
history = Dcnn.fit(train_inputs, train_labels, batch_size=batch_size,
                   epochs=epochs, verbose=1,validation_split=0.10)

ckpt_manager.save()

Epoch 1/5
4725/4725 [==============================] - 2371s 501ms/step - loss: 0.5082 - accuracy: 0.7483 - val_loss: 0.4765 - val_accuracy: 0.7718
Epoch 2/5
4725/4725 [==============================] - 2303s 487ms/step - loss: 0.3450 - accuracy: 0.8490 - val_loss: 0.5274 - val_accuracy: 0.7529
Epoch 3/5
4725/4725 [==============================] - 2311s 489ms/step - loss: 0.1421 - accuracy: 0.9442 - val_loss: 0.8094 - val_accuracy: 0.7203
Epoch 4/5
4155/4725 [=========================>....] - ETA: 4:37 - loss: 0.0578 - accuracy: 0.9781

In [ ]:
results = Dcnn.evaluate(test_inputs, test_labels, batch_size=batch_size)
print(results)

In [ ]:
y_pred_test = Dcnn.predict(test_inputs)
y_pred_test = (y_pred_test > 0.5)
y_pred_test

# **8. Avaliação do modelo**

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(test_labels, y_pred_test)
cm

In [ ]:
sns.heatmap(cm, annot=True)

In [ ]:
history.history.keys()

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model loss progress during training and valitaion')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(['Training loss', 'Validation loss'])

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model accuracy progress during training and valitaion')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(['Training accuracy', 'Validation accuracy'])

# **9. Previsões**

In [ ]:
text = "data science is cool"
text = tokenizer.encode(text)
text

In [ ]:
Dcnn(np.array([text]), training=False).numpy()